In [3]:
#requires Biopython library
#!pip install biopython
WINDOW=15 # range within the search for secondary structure is performed
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import pandas as pd
import regex
import os
import glob
#function for searching for the presence of additional secondary structure elements and 
#export of domain sequences with these elements 
# using associated fasta file and csv file with secondary structure information in DSSP format
def dssp_check(data,fasta,dssp_data):
    records = list(SeqIO.parse(fasta, "fasta"))
    fastas={'id':[],'seq':[],'name':[]}
    for i in range(len(records)):
        fastas['id'].append(records[i].id.split(sep='|')[1])
        fastas['seq'].append(str(records[i].seq))
        fastas['name'].append(str(records[i].id.split(sep='|')[2]))
    fasta_df=pd.DataFrame(fastas)
    protein_ids=[]
    domain_seq=[] #sequences of found domains
    domain_names=[] #Uniprot identifiers
    for index,protein in data.iterrows():
        domains=protein[0].split(sep='\t')
        dssp_ss=list(dssp_data[dssp_data['short_names']==domains[0].split(sep='|')[0]]['ss'])
        protein_id=domains[0].split(sep='|')[0]
        # the search string, E means beta-strand, H - alpha-helix
        pattern_string = 'EEEEEE' #modify accordingly
        #means that no more than two insertions or two substitutions are allowed
        r = regex.compile('(%s){i<=2,s<=2}' % pattern_string) 
        sequence=list(fasta_df[fasta_df['id']==domains[0].split(sep='|')[0]]['seq'])
        name=list(fasta_df[fasta_df['id']==domains[0].split(sep='|')[0]]['name'])
        domains_list={
             'start':[],
             'stop':[],
             'name':[],
        }
        for domain in domains:
            d=domain.split(sep='|')
            if len(d) ==5:
                if d[4] == 'ZnF_C2H2':
                    domains_list['start'].append(d[1])
                    domains_list['stop'].append(d[2])
                    domains_list['name'].append(d[4])
        domains_df=pd.DataFrame(domains_list)
        for znf in range(len(domains_df['start'])):
            if domains_df['name'][znf] == 'ZnF_C2H2': #replace domain name accordingly
                try:
                    if r.search(str(dssp_ss)[(int(domains_df['start'][znf])-WINDOW):(int(domains_df['start'][znf])-2)]):
                        # checking that no other domain is present in search range
                        if znf==0:
                            domain_seq.append(str(sequence)[(int(domains_df['start'][znf])-WINDOW):(int(domains_df['stop'][znf])+5)])
                            domain_names.append(str(name)[2:-2:]+"_"+str(znf))
                        else:
                            if int(domains_df['stop'][znf-1])<(int(domains_df['start'][znf])-WINDOW):
                                domain_seq.append(str(sequence)[(int(domains_df['start'][znf])-WINDOW-5):(int(domains_df['stop'][znf])+5)])
                                domain_names.append((str(name)[2:-2:]+"_"+str(znf)))
                except:
                    None
    print(domain_names)             
    return domain_seq,domain_names
#batch processing of multiple files
#DIRECTORY="txt_files"
for file in glob.glob("*.txt"): 
        data =pd.read_csv(file, skiprows=[0,1,2,3,4,5,6,7,8,9,10,11] )
        name=os.path.basename(file)
        base_name=name.split(sep='.')[0]
        dssp_file=os.path.join(base_name+'.csv')
        fasta=os.path.join(base_name+'.fasta')
        dssp_pred=pd.read_csv(dssp_file)
        domain_sequences,domain_names = dssp_check(data,fasta,dssp_pred)
        dssp_name=os.path.join(base_name+'_dssp_beta_sequences.fasta')
        ofile = open(dssp_name, "w")
        for i in range(len(domain_sequences)):
            ofile.write(">" + domain_names[i] + "\n" +domain_sequences[i] + "\n")
        ofile.close()

['E9FT46_DAPPU_0', 'E9HV70_DAPPU_0', 'E9HZT4_DAPPU_0', 'E9FW32_DAPPU_0', 'E9H5P1_DAPPU_0', 'E9GCT8_DAPPU_0', 'E9GRY7_DAPPU_0', 'E9FUD5_DAPPU_0', 'E9GIR1_DAPPU_0', 'E9GLH2_DAPPU_0', 'E9G6T9_DAPPU_0', 'E9G6T9_DAPPU_3', 'E9HKE0_DAPPU_0', 'E9G141_DAPPU_0', 'E9I4A4_DAPPU_0', 'E9FTE2_DAPPU_0', 'E9I4F8_DAPPU_0', 'E9GVM1_DAPPU_2', 'E9GLD7_DAPPU_0', 'E9G3A7_DAPPU_0', 'E9G3A7_DAPPU_1', 'E9FTE0_DAPPU_0', 'E9HV72_DAPPU_0', 'E9HSC3_DAPPU_0']
['ART1_ORYSJ_0', 'STOP1_ORYSJ_0', 'B1B534_ORYSJ_0', 'Q2QVE1_ORYSJ_0', 'Q7EZ93_ORYSJ_0', 'Q69KN0_ORYSJ_0', 'Q942Y7_ORYSJ_0', 'Q6Z422_ORYSJ_0', 'Q6YXC5_ORYSJ_0', 'Q337G4_ORYSJ_0', 'Q0JF48_ORYSJ_0', 'Q5Z4U6_ORYSJ_0']
['EMF2_ARATH_0', 'FIS2C_ARATH_0', 'IDD13_ARATH_0', 'SRRT_ARATH_0', 'STOP1_ARATH_0', 'STOP2_ARATH_0', 'Q8VZP2_ARATH_0', 'F4HPX2_ARATH_0', 'Q9FM27_ARATH_0']
['A0A0P4W8E7_9EUCA_0', 'A0A0P4WKX8_9EUCA_0', 'A0A0P4W5T3_9EUCA_0', 'A0A0P4W5T3_9EUCA_1', 'A0A0P4W5T3_9EUCA_2', 'A0A0P4W958_9EUCA_0', 'A0A0N7ZBI9_9EUCA_2', 'A0A0P4VTB5_9EUCA_1', 'A0A0P4VTB5_9EUCA_2',

['H9JRM7_BOMMO_0', 'H9JGW0_BOMMO_0', 'H9JGW0_BOMMO_1', 'H9JGW0_BOMMO_2', 'H9JGW1_BOMMO_0', 'H9JGW1_BOMMO_1', 'H9JGW1_BOMMO_2', 'H9JS78_BOMMO_2', 'H9JGZ8_BOMMO_0', 'H9JGZ8_BOMMO_1', 'H9JGZ8_BOMMO_2', 'H9JGZ8_BOMMO_8', 'H9JGZ8_BOMMO_9', 'H9JGZ8_BOMMO_10', 'H9JGZ8_BOMMO_15', 'H9JGZ8_BOMMO_16', 'H9JGZ8_BOMMO_17', 'H9JS06_BOMMO_1', 'H9JS06_BOMMO_2', 'H9JS06_BOMMO_3', 'H9JS06_BOMMO_4', 'H9JGW6_BOMMO_0', 'H9JGW6_BOMMO_1', 'H9JGW6_BOMMO_2', 'H9JF43_BOMMO_0', 'H9JGY9_BOMMO_0', 'H9JGY9_BOMMO_1', 'H9JGY9_BOMMO_2', 'H9J1G3_BOMMO_0', 'H9J7Y4_BOMMO_0', 'H9JGW7_BOMMO_0', 'H9JGW7_BOMMO_1', 'H9JGW7_BOMMO_2', 'H9JGW7_BOMMO_6', 'H9JGW7_BOMMO_8', 'H9JGW7_BOMMO_9', 'H9JGW7_BOMMO_10', 'H9JGX2_BOMMO_0', 'H9JGX2_BOMMO_1', 'H9JGX2_BOMMO_2', 'H9IX45_BOMMO_0', 'H9IX45_BOMMO_3', 'H9IX45_BOMMO_4', 'H9J4F0_BOMMO_0', 'H9IWR2_BOMMO_0', 'H9J4B6_BOMMO_5', 'H9JX72_BOMMO_0', 'H9JGU8_BOMMO_1', 'H9JGU8_BOMMO_2', 'H9JGU8_BOMMO_8', 'H9JGU8_BOMMO_9', 'H9JGU8_BOMMO_10', 'H9IWQ8_BOMMO_0', 'H9JR62_BOMMO_0', 'H9JR62_BOMMO_1', 'H9

['ADNP_MOUSE_5', 'CHAP1_MOUSE_1', 'CHAP1_MOUSE_2', 'CK095_MOUSE_0', 'CK095_MOUSE_1', 'CK095_MOUSE_3', 'IKZF3_MOUSE_4', 'KAISO_MOUSE_0', 'OVOL2_MOUSE_0', 'STPAP_MOUSE_0', 'SUZ12_MOUSE_0', 'WIZ_MOUSE_10', 'Z518A_MOUSE_6', 'ZBT41_MOUSE_1', 'ZBTB4_MOUSE_1', 'ZFA_MOUSE_0', 'ZFHX2_MOUSE_4', 'ZFHX2_MOUSE_14', 'ZFY1_MOUSE_0', 'ZFY2_MOUSE_0', 'ZIC1_MOUSE_0', 'ZIC2_MOUSE_0', 'ZIK1_MOUSE_0', 'ZN292_MOUSE_8', 'ZN318_MOUSE_0', 'ZN318_MOUSE_1', 'ZN335_MOUSE_0', 'ZN382_MOUSE_0', 'ZN507_MOUSE_2', 'ZN618_MOUSE_0', 'ZN639_MOUSE_5', 'ZN668_MOUSE_0', 'ZN710_MOUSE_0', 'ZN711_MOUSE_0', 'E9Q732_MOUSE_0', 'Q3UTQ6_MOUSE_0', 'E9Q5M4_MOUSE_2', 'B1ARH2_MOUSE_0', 'A2AW65_MOUSE_0', 'S4R299_MOUSE_9', 'ZFP28_MOUSE_0', 'G5E869_MOUSE_20', 'G5E869_MOUSE_22', 'ZFX_MOUSE_0', 'ZIC3_MOUSE_0', 'ADNP2_MOUSE_6', 'Q3LR78_MOUSE_2', 'HINFP_MOUSE_3', 'ZN800_MOUSE_6']


In [5]:
zfs=['a','b','c']

In [6]:
len(zfs)

3

In [8]:
for i in range(len(zfs)):
    print(i)

0
1
2
